<table>
    <tr>
        <td colspan=2><strong>International University of La Rioja (UNIR)</strong></td>
    </tr>
    <tr>
        <td colspan=2><strong>Master's Degree in Mathematical Engineering and Computing</strong></td>
    </tr>
    <tr>
        <td>Master's thesis submitted by:</td>
        <td><strong>Andrés Benavides, Diego Salamanca, Yeferson Salamanca.</strong></td>
    </tr>
    <tr>
        <td>Supervisor:</td>
        <td>PhD. Juan Francisco Martínez Cerda</td>
    </tr>
</table>


---


# 04 - Data preprocessing and LSTM dataset construction

This notebook converts the clean merged dataset (`data/processed/merged_processed_data.csv`, produced by `02_merge_data.ipynb`) into **LSTM-ready sliding-window sequences** with a **trend-strength** label (either a **3-class** `BUY / HOLD / SELL` scheme or a **5-class** `BUY_1000USD / BUY_500USD / HOLD / SELL_50% / SELL_100%` scheme), and saves them as `.npy` files for the modeling notebook.

**Inputs**
- `data/processed/merged_processed_data.csv` - numeric, weekday-only, imputed `<TICKER>_Adj Close` columns (and `<TICKER>_Volume` if present).
- `metadata/tickers.json` - sector / macro ticker lists (the number of tickers may change in the future; this notebook is driven entirely by metadata + the columns actually present).
- `metadata/preprocess_config.json` - **all tunable hyperparameters** (see below). Edit this file and rerun the notebook to experiment.

**Outputs** (in `data/datasets/`)
- `train/X.npy`, `train/y.npy`, `val/X.npy`, `val/y.npy`, `test/X.npy`, `test/y.npy`
- `train/ids.npy`, `train/dates.npy` (and val/test) - ticker symbol and as-of date per sample, saved **separately from X** so the model remains time-agnostic but predictions can be mapped back to a specific stock for visualization.
- `scaler.pkl` - the fitted StandardScaler (fitted only on the training split).
- `feature_metadata.json` - a self-describing snapshot of the resolved configuration, feature names, ticker list, array shapes, and class distribution per split.

Feature engineering is implemented once in `src/features.py` and reused in the test analysis notebook (05), so recommendations on the test set are constructed in exactly the same way as the training data.

**Workflow**
1. Load configuration + merged data, resolve sector/macro tickers from metadata.
2. **Feature engineering** - transform raw prices into trend-based features per sector ticker (log returns, moving-average deviations, RSI), plus shared macro/volatility context.
3. **Labels** - the forward `prediction_horizon`-day return per ticker, binned into the active scheme's classes (3 or 5) according to the configuration thresholds.
4. **Chronological split** (train/val/test) by date - no shuffling (time series).
5. **StandardScaler scaling** fitted only on the training split (no information leakage), applied to all splits.
6. **Sliding windows** of `window_size` days generated *within* each split (so no window crosses a boundary), per ticker, then stacked across all sector tickers into a single pooled dataset.
7. Save the `.npy` files + scaler + metadata.

**Configuration keys (`metadata/preprocess_config.json`)**
- `window_size` - time steps per sequence (e.g. 60 days -> classify the day after the window).
- `prediction_horizon` - how many days ahead the label looks (currently 10).
- `label_scheme` - which scheme to build in this run: `"3class"` or `"5class"`.
- `label_schemes.<name>` - per scheme: `classes` (display order, positive->negative), `class_mapping` (integer codes, bearish->bullish ordinal order), and `thresholds` (named forward-return cutoffs; the notebook sorts them into ascending bin edges).
- `feature_windows.return_windows` / `.ma_windows` / `.rsi_window` / `.volume_ma_window` - rolling windows for generated features.
- `features.*` - enable/disable each generated feature family (`use_log_returns`, `use_ma_ratios`, `use_rsi`, `use_volume_ratio`, `use_realized_vol`).
- `macro_windows.zscore_window` / `.ma_window` - windows for the macro `zscore` and `log_ma_ratio` transformations.
- `macro_features` - one transformation or a list per macro ticker. Options: `"level"` (raw value), `"return"` (daily percentage change), `"logret_{n}d"` (`n`-day log return), `"diff_5d"` (5-day level change, e.g. yield points for `^TNX`), `"zscore"` (rolling level z-score), and `"log_ma_ratio"` (`log(level / MA)`).
- `split_ratios.train` / `.val` / `.test` - chronological split fractions (must sum to 1.0).
- `scaler` - currently `"standard"`.
- `random_seed`, `input_csv`, `output_dir` - reproducibility and relative paths.

**Label schemes**
- *3 classes* `[BUY, HOLD, SELL]` with 2 thresholds (`buy`, `sell`).
- *5 classes* `[BUY_1000USD, BUY_500USD, HOLD, SELL_50%, SELL_100%]` with 4 thresholds (`strong_buy`, `buy`, `sell`, `strong_sell`) so signal strength determines position size. Both are ordinal (higher integer code = more bullish), so the modeling/inference notebooks adapt automatically by reading `class_mapping`.

**Feature design rationale**
- *Sector `Adj Close`* -> trend features (multi-window log returns, 10- and 50-day `log(price/MA)` deviations, RSI-14, `realized_vol`) and the label source.
- *Sector `Volume`* -> `log_vol_ratio` = `log1p(volume) - log1p(MA(volume))`, an indicator of relative activity.
- *Macro `Adj Close`* (`^VIX`, `DX-Y.NYB`, `^TNX`) -> shared market context with series-specific transformations (e.g. `^VIX`: z-score + `log(VIX/MA)` + `logret_5d`; `DX-Y.NYB`: `logret_5d` and `logret_20d`; `^TNX`: `diff_5d` in yield points).

## 1. Imports and paths

In [15]:
import importlib
import json
import logging
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("preprocess_data")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
METADATA_PATH = PROJECT_ROOT / "metadata" / "tickers.json"
CONFIG_PATH = PROJECT_ROOT / "metadata" / "preprocess_config.json"

# Shared feature-engineering module (also used by the inference notebook 05).
# Reload so edits to src/features.py are picked up without restarting the kernel.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import features as fe
importlib.reload(fe)

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"METADATA_PATH : {METADATA_PATH}")
print(f"CONFIG_PATH   : {CONFIG_PATH}")

PROJECT_ROOT  : /Applications/TFM_LSTM
METADATA_PATH : /Applications/TFM_LSTM/metadata/tickers.json
CONFIG_PATH   : /Applications/TFM_LSTM/metadata/preprocess_config.json


## 2. Load configuration

Load all tunable hyperparameters from `metadata/preprocess_config.json`. To experiment, edit the JSON file and rerun from here.

In [16]:
if CONFIG_PATH.exists():
    with CONFIG_PATH.open("r", encoding="utf-8") as f:
        cfg = json.load(f)
    logger.info("Loaded config from %s", CONFIG_PATH)
else:
    cfg = {}
    logger.warning("No config at %s - using built-in defaults", CONFIG_PATH)


# Resolve / validate the values used throughout the notebook.
WINDOW_SIZE = int(cfg["window_size"])
HORIZON = int(cfg["prediction_horizon"])

# --- Label scheme (3-class or 5-class) ---------------------------------------
# Pick the active scheme via cfg["label_scheme"]; thresholds for every scheme
# live in cfg["label_schemes"] so you can switch by editing one key and re-running.
SCHEME_NAME = cfg["label_scheme"]
scheme = cfg["label_schemes"][SCHEME_NAME]
CLASS_MAP = {k: int(v) for k, v in scheme["class_mapping"].items()}
# Class names ordered most-bearish -> most-bullish (by their integer code).
ORDERED_CLASSES = [k for k, _ in sorted(CLASS_MAP.items(), key=lambda kv: kv[1])]
CLASSES_DISPLAY = scheme.get("classes", ORDERED_CLASSES)

# --- Label mode (absolute vs volatility-scaled) ------------------------------
# "absolute"   -> bin the raw HORIZON-day forward return against fixed return edges.
# "vol_scaled" -> bin the volatility-normalized return z = r / (sigma_daily *
#                 sqrt(horizon)), so a class means the same risk-adjusted move
#                 for every ticker. sigma uses a trailing VOL_WINDOW of daily
#                 returns (no look-ahead). Edges are then in units of sigma.
LABEL_MODE = cfg.get("label_mode")
VOL_WINDOW = int(cfg.get("vol_window"))
# Round-trip transaction-cost buffer (return units): a move must beat both
# volatility and cost to count as a signal (reference pipeline's COST_BPS).
COST = float(cfg.get("cost", 0.0))
THRESHOLD_KEY = "thresholds_vol" if LABEL_MODE == "vol_scaled" else "thresholds"
EDGE_UNITS = "sigma" if LABEL_MODE == "vol_scaled" else "return"
# Ascending bin edges built from the named thresholds (len = n_classes - 1).
EDGES = sorted(float(v) for v in scheme[THRESHOLD_KEY].values())
LABEL_THRESHOLDS = dict(scheme[THRESHOLD_KEY])

# --- Feature windows ---------------------------------------------------------
RETURN_WINDOWS = [int(n) for n in cfg["feature_windows"]["return_windows"]]
MA_WINDOWS = [int(n) for n in cfg["feature_windows"]["ma_windows"]]
RSI_WINDOW = int(cfg["feature_windows"]["rsi_window"])
VOL_MA_WINDOW = int(cfg["feature_windows"]["volume_ma_window"])
FEATURE_FLAGS = cfg["features"]
MACRO_FEATURES = cfg["macro_features"]
MACRO_WINDOWS = cfg.get("macro_windows", {"zscore_window": 20, "ma_window": 20})
SPLIT_RATIOS = cfg["split_ratios"]
SEED = int(cfg["random_seed"])

np.random.seed(SEED)

INPUT_CSV = PROJECT_ROOT / cfg["input_csv"]
OUTPUT_DIR = PROJECT_ROOT / cfg["output_dir"]

ratio_sum = SPLIT_RATIOS["train"] + SPLIT_RATIOS["val"] + SPLIT_RATIOS["test"]
assert abs(ratio_sum - 1.0) < 1e-6, f"split_ratios must sum to 1.0 (got {ratio_sum})"
assert LABEL_MODE in ("absolute", "vol_scaled"), (
    f"label_mode must be 'absolute' or 'vol_scaled' (got '{LABEL_MODE}')"
)
assert len(EDGES) == len(ORDERED_CLASSES) - 1, (
    f"scheme '{SCHEME_NAME}' must define exactly len(classes)-1 '{THRESHOLD_KEY}' "
    f"(got {len(EDGES)} edges for {len(ORDERED_CLASSES)} classes)"
)
assert EDGES == sorted(EDGES), "threshold edges must be strictly ascending"

print(f"Active label scheme : {SCHEME_NAME}")
print(f"Label mode          : {LABEL_MODE}" + (f" (vol_window={VOL_WINDOW})" if LABEL_MODE == "vol_scaled" else ""))
print(f"Transaction cost    : {COST} (return units)")
print(f"Classes (bearish->bullish): {ORDERED_CLASSES}")
print(f"Bin edges ({EDGE_UNITS} units): {EDGES}")
print("Resolved configuration:")
print(json.dumps(cfg, indent=2))
print(f"\nINPUT_CSV  : {INPUT_CSV}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

14:54:24 | INFO | Loaded config from /Applications/TFM_LSTM/metadata/preprocess_config.json


Active label scheme : 5class
Label mode          : vol_scaled (vol_window=20)
Transaction cost    : 0.001 (return units)
Classes (bearish->bullish): ['SELL_100%', 'SELL_50%', 'HOLD', 'BUY_500USD', 'BUY_1000USD']
Bin edges (sigma units): [-2.5, -1.2, 1.2, 2.5]
Resolved configuration:
{
  "window_size": 60,
  "prediction_horizon": 10,
  "label_scheme": "5class",
  "label_mode": "vol_scaled",
  "vol_window": 20,
  "feature_windows": {
    "return_windows": [
      1,
      5,
      10,
      20,
      50
    ],
    "ma_windows": [
      10,
      50
    ],
    "rsi_window": 14,
    "volume_ma_window": 20
  },
  "features": {
    "use_log_returns": true,
    "use_ma_ratios": true,
    "use_rsi": true,
    "use_volume_ratio": true,
    "use_realized_vol": true
  },
  "macro_windows": {
    "zscore_window": 20,
    "ma_window": 20
  },
  "macro_features": {
    "^VIX": [
      "logret_5d"
    ],
    "DX-Y.NYB": [
      "logret_5d"
    ],
    "^TNX": [
      "logret_5d"
    ]
  },
  "label_sc

## 3. Load merged data and resolve stock tickers

Load `merged_processed_data.csv` (Date index) and resolve which sector / macro tickers to use from `metadata/tickers.json`. Only tickers that actually have an `Adj Close` column in the CSV are retained, so a changing ticker set in the metadata is handled automatically. We also detect whether `<TICKER>_Volume` columns are present; if not, the volume feature is skipped.

In [ ]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Missing input CSV: {INPUT_CSV}. Run 02_merge_data.ipynb first."
    )
if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Missing metadata file: {METADATA_PATH}.")

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

data = pd.read_csv(INPUT_CSV, index_col=0, parse_dates=True).sort_index()
data.index.name = "Date"

meta_sector_tickers = [t for tk in metadata["sectors"].values() for t in tk.keys()]
meta_macro_tickers = list(metadata.get("macro", {}).keys())

# Keep only tickers that actually have an Adj Close column in the CSV.
sector_tickers = [t for t in meta_sector_tickers if f"{t}_Adj Close" in data.columns]
macro_tickers = [t for t in meta_macro_tickers if f"{t}_Adj Close" in data.columns]

missing_sector = [t for t in meta_sector_tickers if t not in sector_tickers]
missing_macro = [t for t in meta_macro_tickers if t not in macro_tickers]

# Volume is optional; only used if at least one sector ticker has a Volume column.
has_volume = any(f"{t}_Volume" in data.columns for t in sector_tickers)

# Only keep macro features whose ticker is actually available.
macro_features_active = {
    mt: rep for mt, rep in MACRO_FEATURES.items() if mt in macro_tickers
}

print(f"Data shape          : {data.shape}")
print(f"Date range          : {data.index.min().date()} -> {data.index.max().date()}")
print(f"Sector tickers used : {len(sector_tickers)} / {len(meta_sector_tickers)}")
print(f"Macro tickers used  : {len(macro_tickers)} / {len(meta_macro_tickers)}")
print(f"Volume columns?     : {has_volume}")
print(f"Macro features      : {macro_features_active}")
if missing_sector:
    logger.warning("Sector tickers in metadata but not in CSV: %s", missing_sector)
if missing_macro:
    logger.warning("Macro tickers in metadata but not in CSV: %s", missing_macro)
if FEATURE_FLAGS.get("use_volume_ratio", True) and not has_volume:
    logger.warning(
        "use_volume_ratio is True but no <TICKER>_Volume columns found - "
        "re-run 02_merge_data.ipynb with Volume to enable this feature. Skipping it."
    )

Data shape          : (3018, 53)
Date range          : 2014-05-30 -> 2026-05-29
Sector tickers used : 25 / 25
Macro tickers used  : 3 / 3
Volume columns?     : True
Macro features      : {'^VIX': ['logret_5d'], 'DX-Y.NYB': ['logret_5d'], '^TNX': ['logret_5d']}


## 4. Feature engineering and labels

For each sector ticker, we transform raw `Adj Close` (and `Volume`) into trend-based features and construct the classification label from the **forward `HORIZON`-day return**:

- `logret_{n}d` - `n`-day log return (`log(price) - log(price.shift(n))`) for each `n` in `return_windows` (default 2, 5, 10, 20, 50).
- `log_price_ma{n}` - log deviation from the moving average: `log(price / MA_n)` for each `n` in `ma_windows` (default 10, 50).
- `rsi_{rsi_window}` - Wilder's RSI momentum oscillator (default 14).
- `log_vol_ratio` - `log1p(Volume) - log1p(MA_volume)`; only if Volume is available.
- `realized_vol` - trailing realized volatility (`std(logret_1d, vol_window) * sqrt(horizon)`), the same quantity that scales the `vol_scaled` labels (no look-ahead).

The **shared macro features** (the same values broadcast to each ticker's sample) come from `macro_features` in the configuration. Each macro ticker can have one or more transformations: `level`, `return`, `logret_{n}d`, `diff_5d` (yield change in points), `zscore` (VIX regime), or `log_ma_ratio` (`log(level/MA)`). The current configuration uses different representations per series instead of applying the same `logret_5d` to all of them.

The **label** for day `t`: `r = log(AdjClose[t+HORIZON]) - log(AdjClose[t])` is assigned to one of the active scheme's bins using ascending threshold edges (e.g. 3 classes: `> buy` -> BUY, `< sell` -> SELL, otherwise HOLD; 5 classes further subdivide BUY/SELL using `strong_buy`/`strong_sell`). This label is associated with the **last day of each window**, so a 60-day window classifies the following day.

Each sector ticker produces the same ordered feature vector (generic per-ticker feature names + fixed macro names), allowing us to stack all tickers into a single dataset later.

In [18]:
# Feature engineering / labelling now live in the shared `src/features.py` module
# (imported as `fe`) so notebook 03 and the inference notebook 05 derive features
# identically. The functions take their parameters explicitly from the config.

macro_df = fe.build_macro_features(data, macro_features_active, MACRO_WINDOWS)

# Per-ticker (features, labels). Feature column order is identical across tickers.
ticker_frames: dict[str, pd.DataFrame] = {}
ticker_labels: dict[str, pd.Series] = {}
for ticker in sector_tickers:
    ticker_frames[ticker] = fe.build_ticker_frame(
        data, ticker, macro_df, FEATURE_FLAGS,
        RETURN_WINDOWS, MA_WINDOWS, RSI_WINDOW, VOL_MA_WINDOW, has_volume,
        vol_window=VOL_WINDOW, horizon=HORIZON,
    )
    ticker_labels[ticker] = fe.build_labels(
        data, ticker, HORIZON, ORDERED_CLASSES, EDGES, CLASS_MAP,
        mode=LABEL_MODE, vol_window=VOL_WINDOW, cost=COST,
    )

FEATURE_NAMES = list(ticker_frames[sector_tickers[0]].columns)
N_FEATURES = len(FEATURE_NAMES)

# Sanity check: every ticker must expose the same ordered feature set.
for ticker in sector_tickers:
    assert list(ticker_frames[ticker].columns) == FEATURE_NAMES, (
        f"Feature columns for {ticker} differ from the reference order"
    )

print(f"Features per timestep ({N_FEATURES}):")
for name in FEATURE_NAMES:
    print(f"  - {name}")
print("\nExample (AAPL-like) tail of engineered features:")
ticker_frames[sector_tickers[0]].join(
    ticker_labels[sector_tickers[0]].rename("label")
).tail()

Features per timestep (13):
  - logret_1d
  - logret_5d
  - logret_10d
  - logret_20d
  - logret_50d
  - log_price_ma10
  - log_price_ma50
  - rsi_14
  - log_vol_ratio
  - realized_vol
  - macro::^VIX::logret5
  - macro::DX-Y.NYB::logret5
  - macro::^TNX::logret5

Example (AAPL-like) tail of engineered features:


,logret_1d,logret_5d,logret_10d,logret_20d,logret_50d,log_price_ma10,log_price_ma50,rsi_14,log_vol_ratio,realized_vol,macro::^VIX::logret5,macro::DX-Y.NYB::logret5,macro::^TNX::logret5,label
Date,,,,,,,,,,,,,,
2026-05-22,0.012480,0.028210,0.052415,0.131339,0.189440,0.029756,0.133012,78.630153,-0.102823,0.036806,-0.098571,0.000504,-0.008085,NaN
2026-05-26,-0.001588,0.034614,0.052091,0.142561,0.210151,0.022961,0.127110,77.634822,-0.015005,0.034479,-0.046520,0.002019,-0.028523,NaN
2026-05-27,0.008140,0.038967,0.053013,0.139183,0.207554,0.025789,0.130968,79.100027,0.023760,0.034335,-0.103148,-0.000907,-0.040670,NaN
2026-05-28,0.005326,0.033382,0.044628,0.146506,0.207318,0.026622,0.132011,80.028208,-0.039344,0.033714,-0.102561,-0.000909,-0.025924,NaN
2026-05-29,-0.001441,0.022917,0.045398,0.140707,0.222895,0.020640,0.126027,79.003888,0.355162,0.034229,-0.089836,-0.002827,-0.029430,NaN


## 5. Chronological split and standard scaling

Time series must be split **by time**, never shuffled. We calculate two cutoff dates from the full date range using `split_ratios`, yielding contiguous train / val / test date ranges shared by all tickers.

The StandardScaler is then fitted **only on training rows** (pooled across all tickers, per feature column) and applied to val / test; this prevents future scaling statistics from leaking backward.

In [19]:
all_dates = data.index
n_dates = len(all_dates)
train_end_i = int(n_dates * SPLIT_RATIOS["train"])
val_end_i = train_end_i + int(n_dates * SPLIT_RATIOS["val"])

train_dates = all_dates[:train_end_i]
val_dates = all_dates[train_end_i:val_end_i]
test_dates = all_dates[val_end_i:]

split_date_sets = {
    "train": set(train_dates),
    "val": set(val_dates),
    "test": set(test_dates),
}

print("Chronological split (by date):")
print(f"  train : {train_dates.min().date()} -> {train_dates.max().date()} ({len(train_dates)} days)")
print(f"  val   : {val_dates.min().date()} -> {val_dates.max().date()} ({len(val_dates)} days)")
print(f"  test  : {test_dates.min().date()} -> {test_dates.max().date()} ({len(test_dates)} days)")

# Clean each ticker (drop warmup / label NaNs) and remember the row -> date mapping.
clean_frames: dict[str, pd.DataFrame] = {}
clean_labels: dict[str, pd.Series] = {}
for ticker in sector_tickers:
    feats = ticker_frames[ticker]
    labels = ticker_labels[ticker]
    valid_idx = feats.dropna().index.intersection(labels.dropna().index)
    clean_frames[ticker] = feats.loc[valid_idx]
    clean_labels[ticker] = labels.loc[valid_idx].astype(int)

# Fit the scaler on TRAIN rows only, pooled across all tickers.
train_rows = [
    df.loc[df.index.isin(split_date_sets["train"])].values
    for df in clean_frames.values()
]
train_matrix = np.vstack([r for r in train_rows if len(r) > 0])

if cfg["scaler"] == "minmax":
    scaler = MinMaxScaler()
elif cfg["scaler"] == "standard":
    # Zero-mean / unit-variance. Preferred for LSTMs on return-style features:
    # unlike Min-Max it is far less sensitive to single extreme outliers.
    scaler = StandardScaler()
else:
    raise ValueError(f"Unsupported scaler '{cfg['scaler']}' (use 'minmax' or 'standard')")

scaler.fit(train_matrix)

print(f"\nScaler ({cfg['scaler']}) fit on {train_matrix.shape[0]} pooled training rows x {train_matrix.shape[1]} features.")
if isinstance(scaler, MinMaxScaler):
    print(f"Per-feature train min : {np.round(scaler.data_min_, 4)}")
    print(f"Per-feature train max : {np.round(scaler.data_max_, 4)}")
else:
    print(f"Per-feature train mean : {np.round(scaler.mean_, 4)}")
    print(f"Per-feature train std  : {np.round(scaler.scale_, 4)}")

Chronological split (by date):
  train : 2014-05-30 -> 2024-05-23 (2514 days)
  val   : 2024-05-24 -> 2025-05-27 (251 days)
  test  : 2025-05-28 -> 2026-05-29 (253 days)

Scaler (standard) fit on 61600 pooled training rows x 13 features.
Per-feature train mean : [ 6.00000e-04  2.80000e-03  5.60000e-03  1.10000e-02  2.75000e-02
  2.20000e-03  1.19000e-02  5.30474e+01 -5.50000e-02  5.55000e-02
 -6.00000e-04  5.00000e-04  1.20000e-03]
Per-feature train std  : [2.02000e-02 4.38000e-02 6.18000e-02 8.73000e-02 1.34200e-01 3.31000e-02
 7.76000e-02 1.22709e+01 3.41800e-01 3.18000e-02 1.60700e-01 9.80000e-03
 6.30000e-02]


## 6. Sliding-window sequence generation

For each ticker and each split, we take the contiguous clean rows (sorted by date) belonging to that split, scale them with the fitted scaler, and slide a window of length `WINDOW_SIZE` over them. Each window becomes a sample of shape `(WINDOW_SIZE, N_FEATURES)`, and its label is the class of the **last day of the window** (i.e. it classifies the day immediately after the window). Windows are built independently within each split, so none crosses a train/val/test boundary. Finally, we stack each ticker's windows into a single pooled dataset per split.

In [20]:
splits = {"train": train_dates, "val": val_dates, "test": test_dates}
X_data: dict[str, np.ndarray] = {}
y_data: dict[str, np.ndarray] = {}
# Per-sample alignment arrays (metadata only - NEVER fed to the model):
#   ids_data[split]   -> ticker symbol that produced each window
#   dates_data[split] -> "as-of" date (last/most-recent input day of each window)
ids_data: dict[str, np.ndarray] = {}
dates_data: dict[str, np.ndarray] = {}

for split_name, split_index in splits.items():
    date_set = split_date_sets[split_name]
    x_parts, y_parts, id_parts, date_parts = [], [], [], []
    for ticker in sector_tickers:
        df = clean_frames[ticker]
        lbl = clean_labels[ticker]
        mask = df.index.isin(date_set)
        sub = df.loc[mask]
        if sub.empty:
            continue
        scaled = scaler.transform(sub.values)
        sub_dates = sub.index.strftime("%Y-%m-%d").to_numpy()
        xw, yw, dw = fe.make_windows(scaled, lbl.loc[mask].values, sub_dates, WINDOW_SIZE)
        if len(xw):
            x_parts.append(xw)
            y_parts.append(yw)
            date_parts.append(dw)
            id_parts.append(np.full(len(xw), ticker, dtype=object))
    if x_parts:
        X_data[split_name] = np.concatenate(x_parts, axis=0)
        y_data[split_name] = np.concatenate(y_parts, axis=0)
        ids_data[split_name] = np.concatenate(id_parts, axis=0)
        dates_data[split_name] = np.concatenate(date_parts, axis=0)
    else:
        X_data[split_name] = np.empty((0, WINDOW_SIZE, N_FEATURES), dtype=np.float32)
        y_data[split_name] = np.empty((0,), dtype=np.int64)
        ids_data[split_name] = np.empty((0,), dtype=object)
        dates_data[split_name] = np.empty((0,), dtype=object)

inv_class_map = {v: k for k, v in CLASS_MAP.items()}
print("Generated sequences (X = features only; ids/dates kept separately for mapping):")
for split_name in splits:
    X, y = X_data[split_name], y_data[split_name]
    dist = {inv_class_map[c]: int((y == c).sum()) for c in sorted(inv_class_map)}
    n_tickers = len(np.unique(ids_data[split_name])) if len(ids_data[split_name]) else 0
    print(f"  {split_name:<5s} X={X.shape}  y={y.shape}  ids={ids_data[split_name].shape} ({n_tickers} tickers)  classes={dist}")

# --- Class-balance report (counts + %) to calibrate the threshold edges -------
# With LABEL_MODE='vol_scaled' the edges are in sigma units; nudge them in the
# config until these proportions match the balance you want per class.
print(f"\nClass distribution ({LABEL_MODE}, edges {EDGES} {EDGE_UNITS} units):")
header = "  " + f"{'split':<8s}" + "".join(f"{inv_class_map[c]:>16s}" for c in sorted(inv_class_map)) + f"{'total':>10s}"
print(header)
y_all_parts = [y_data[s] for s in splits if len(y_data[s])]
y_all = np.concatenate(y_all_parts) if y_all_parts else np.empty((0,), dtype=np.int64)
for split_name, y in list((s, y_data[s]) for s in splits) + [("ALL", y_all)]:
    total = int(len(y))
    cells = ""
    for c in sorted(inv_class_map):
        n = int((y == c).sum())
        pct = (100.0 * n / total) if total else 0.0
        cells += f"{f'{n} ({pct:.1f}%)':>16s}"
    print(f"  {split_name:<8s}{cells}{total:>10d}")

Generated sequences (X = features only; ids/dates kept separately for mapping):
  train X=(60125, 60, 13)  y=(60125,)  ids=(60125,) (25 tickers)  classes={'SELL_100%': 1298, 'SELL_50%': 5049, 'HOLD': 45686, 'BUY_500USD': 6960, 'BUY_1000USD': 1132}
  val   X=(4800, 60, 13)  y=(4800,)  ids=(4800,) (25 tickers)  classes={'SELL_100%': 156, 'SELL_50%': 527, 'HOLD': 3587, 'BUY_500USD': 468, 'BUY_1000USD': 62}
  test  X=(4600, 60, 13)  y=(4600,)  ids=(4600,) (25 tickers)  classes={'SELL_100%': 46, 'SELL_50%': 303, 'HOLD': 3523, 'BUY_500USD': 588, 'BUY_1000USD': 140}

Class distribution (vol_scaled, edges [-2.5, -1.2, 1.2, 2.5] sigma units):
  split          SELL_100%        SELL_50%            HOLD      BUY_500USD     BUY_1000USD     total
  train        1298 (2.2%)     5049 (8.4%)   45686 (76.0%)    6960 (11.6%)     1132 (1.9%)     60125
  val           156 (3.2%)     527 (11.0%)    3587 (74.7%)      468 (9.8%)       62 (1.3%)      4800
  test           46 (1.0%)      303 (6.6%)    3523 (76.

## 7. Save datasets, scaler, and metadata

Write the `.npy` arrays to `data/datasets/{train,val,test}/` - `X`/`y` for the model plus `ids`/`dates` for per-ticker mapping - save the fitted scaler (`scaler.pkl`), and save a `feature_metadata.json` capturing the resolved configuration plus feature names, ticker list, array shapes, and class distribution per split, so each run is self-describing and reproducible by the modeling and test analysis notebooks.

In [21]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split_name in splits:
    split_dir = OUTPUT_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    np.save(split_dir / "X.npy", X_data[split_name])
    np.save(split_dir / "y.npy", y_data[split_name])
    # Alignment arrays for per-ticker analysis / display (not used for training).
    np.save(split_dir / "ids.npy", ids_data[split_name])
    np.save(split_dir / "dates.npy", dates_data[split_name])
    logger.info(
        "Saved %s: X=%s y=%s ids=%s dates=%s",
        split_name, X_data[split_name].shape, y_data[split_name].shape,
        ids_data[split_name].shape, dates_data[split_name].shape,
    )

SCALER_PATH = OUTPUT_DIR / "scaler.pkl"
joblib.dump(scaler, SCALER_PATH)

class_distribution = {
    split_name: {
        inv_class_map[c]: int((y_data[split_name] == c).sum())
        for c in sorted(inv_class_map)
    }
    for split_name in splits
}

feature_metadata = {
    "config": cfg,
    "feature_names": FEATURE_NAMES,
    "n_features": N_FEATURES,
    "window_size": WINDOW_SIZE,
    "prediction_horizon": HORIZON,
    "label_scheme": SCHEME_NAME,
    "label_mode": LABEL_MODE,
    "vol_window": VOL_WINDOW,
    "classes_display": CLASSES_DISPLAY,
    "ordered_classes": ORDERED_CLASSES,
    "label_thresholds": LABEL_THRESHOLDS,
    "label_edges": EDGES,
    "label_edge_units": EDGE_UNITS,
    "class_mapping": CLASS_MAP,
    "sector_tickers_used": sector_tickers,
    "macro_tickers_used": macro_tickers,
    "macro_features_active": macro_features_active,
    "macro_windows": MACRO_WINDOWS,
    "has_volume": bool(has_volume),
    "per_sample_arrays": {
        "ids.npy": "ticker symbol for each sample (for per-ticker analysis)",
        "dates.npy": "as-of date (last input day) for each sample; not a feature",
    },
    "input_csv": str(INPUT_CSV),
    "date_range": {
        "train": [str(train_dates.min().date()), str(train_dates.max().date())],
        "val": [str(val_dates.min().date()), str(val_dates.max().date())],
        "test": [str(test_dates.min().date()), str(test_dates.max().date())],
    },
    "shapes": {s: list(X_data[s].shape) for s in splits},
    "class_distribution": class_distribution,
}

METADATA_OUT = OUTPUT_DIR / "feature_metadata.json"
METADATA_OUT.write_text(json.dumps(feature_metadata, indent=2), encoding="utf-8")

print("=" * 60)
print("DATASETS SAVED")
print("=" * 60)
for split_name in splits:
    print(f"  {split_name:<5s} -> {OUTPUT_DIR / split_name} (X={X_data[split_name].shape}, y={y_data[split_name].shape})")
print(f"  scaler   -> {SCALER_PATH}")
print(f"  metadata -> {METADATA_OUT}")
print("-" * 60)
print("Class distribution per split:")
for split_name, dist in class_distribution.items():
    total = sum(dist.values())
    pretty = {k: f"{v} ({100 * v / total:.1f}%)" if total else f"{v}" for k, v in dist.items()}
    print(f"  {split_name:<5s}: {pretty}")

14:54:25 | INFO | Saved train: X=(60125, 60, 13) y=(60125,) ids=(60125,) dates=(60125,)
14:54:25 | INFO | Saved val: X=(4800, 60, 13) y=(4800,) ids=(4800,) dates=(4800,)
14:54:25 | INFO | Saved test: X=(4600, 60, 13) y=(4600,) ids=(4600,) dates=(4600,)


DATASETS SAVED
  train -> /Applications/TFM_LSTM/data/datasets/train (X=(60125, 60, 13), y=(60125,))
  val   -> /Applications/TFM_LSTM/data/datasets/val (X=(4800, 60, 13), y=(4800,))
  test  -> /Applications/TFM_LSTM/data/datasets/test (X=(4600, 60, 13), y=(4600,))
  scaler   -> /Applications/TFM_LSTM/data/datasets/scaler.pkl
  metadata -> /Applications/TFM_LSTM/data/datasets/feature_metadata.json
------------------------------------------------------------
Class distribution per split:
  train: {'SELL_100%': '1298 (2.2%)', 'SELL_50%': '5049 (8.4%)', 'HOLD': '45686 (76.0%)', 'BUY_500USD': '6960 (11.6%)', 'BUY_1000USD': '1132 (1.9%)'}
  val  : {'SELL_100%': '156 (3.2%)', 'SELL_50%': '527 (11.0%)', 'HOLD': '3587 (74.7%)', 'BUY_500USD': '468 (9.8%)', 'BUY_1000USD': '62 (1.3%)'}
  test : {'SELL_100%': '46 (1.0%)', 'SELL_50%': '303 (6.6%)', 'HOLD': '3523 (76.6%)', 'BUY_500USD': '588 (12.8%)', 'BUY_1000USD': '140 (3.0%)'}
